In [1]:
#chromadb


In [2]:
#Building an RAG System with langchain and chromadb

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings


#vector stroes
from langchain_community.vectorstores import chroma

#utility imports
import numpy as np
from typing import List,Dict


c:\Users\PARAS\Downloads\krish-naik-new-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
#Creating sample documnets
sample_docs=[
  """
    MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data without being explicitly programmed. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled data. Machine learning is widely used in applications like recommendation systems, fraud detection, predictive analytics, and medical diagnosis, making it a cornerstone of modern AI.",
    """,
    """
    DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw data. Unlike traditional machine learning, which often requires manual feature engineering, deep learning models can process unstructured data such as images, audio, and text directly, extracting hierarchical features through successive layers. These networks, often containing millions of parameters, are trained using large datasets and powerful computational resources, enabling breakthroughs in areas like computer vision, natural language processing, autonomous driving, and speech recognition. Deep learning has revolutionized AI by achieving human-level or even superhuman performance in tasks that were previously considered extremely difficult for machines.",
    """,

    """
    NeuralNetworks: "Neural networks are computational models inspired by the structure and functioning of the human brain. They consist of interconnected nodes, called neurons, organized into layers: an input layer, one or more hidden layers, and an output layer. Each neuron applies mathematical transformations to its inputs and passes the result forward, allowing the network to learn complex, non-linear relationships in data. Neural networks can be shallow, with only a few layers, or deep, with many hidden layers, forming the basis of deep learning. Variants such as convolutional neural networks (CNNs) excel at image recognition by capturing spatial hierarchies, while recurrent neural networks (RNNs) are designed to handle sequential data like text or time series. Neural networks are fundamental to modern AI, enabling systems to recognize patterns, make predictions, and adapt to new information."
  """
]


In [6]:
sample_docs

['\n    MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data without being explicitly programmed. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled data. Machine learning is widely used in applications like recommendation systems, fraud detection, predictive analytics, and medical diagnosis, making it a cornerstone of modern AI.",\n    ',
 '\n    DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw data. Unlike traditional machine learni

In [78]:
import tempfile
temp_dir=tempfile.mkdtemp()
for i,doc in enumerate(sample_docs):
    with open(f'doc_{i}.txt','w') as f:
        f.write(doc)
print(f"Sample documents created in: {temp_dir}")

Sample documents created in: C:\Users\PARAS\AppData\Local\Temp\tmpfd4h6oi8


In [79]:
import tempfile
temp_dir=tempfile.mkdtemp()
for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt",'w') as f:
        f.write(doc)
    

In [10]:
#2-documents loading
from langchain_community.document_loaders import DirectoryLoader,TextLoader
loader=DirectoryLoader(
    'data',
    glob='*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)
documents=loader.load()

print(f"the length of the document is {len(documents)}")
print(f"The first document is {documents[0]}")
print(documents[0].page_content[:200]+"....")

the length of the document is 3
The first document is page_content='
    MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data without being explicitly programmed. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled data. Machine learning is widely used in applications like recommendation systems, fraud detection, predictive analytics, and medical diagnosis, making it a cornerstone of modern AI.",
    ' metadata={'source': 'data\\doc_0.txt'}

    MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable c

In [84]:
#Document splitting will be done now 
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n",'\n','.',' ']
)
chunks=text_splitter.split_documents(documents)



In [12]:
text_splitter

In [13]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data without'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='from data without being explicitly programmed'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='in unlabeled data'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. Ma

In [14]:
#  Embedding models

In [15]:
sample_text="Machine learning is fascinating"

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

#initialize z simple embddings model
embeddings=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',

)
embeddings


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [17]:
vector_text=embeddings.aembed_query(sample_text)

In [18]:
vector_text

<coroutine object Embeddings.aembed_query at 0x00000261E9520120>

In [19]:
#initialized the crhomadb vector_stores
!pip install vectorstores

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [20]:
!pip install chromadb
!pip install langchain-community
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [21]:
pip install chroma

Note: you may need to restart the kernel to use updated packages.


c:\Users\PARAS\Downloads\krish-naik-new-course\.venv\Scripts\python.exe: No module named pip


In [22]:


# Initialize embeddings
embeddings = HuggingFaceEmbeddings()
persist_directory="./chroma_db"
# Create or load a persistent Chroma database
vector_store = chroma.Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name='rag_collection'
)

In [23]:
print(f"The vector_store that has been created  is {vector_store._collection.count()} vectors")

The vector_store that has been created  is 103 vectors


In [24]:
print(f"In here the persist director is {persist_directory}")

In here the persist director is ./chroma_db


In [25]:
#Test similarity search
query="What are the types of machine learning?"

In [26]:
similar_docs=vector_store.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled')]

In [27]:
query="What is deep learning?"
similar_docs=vector_store.similarity_search(query,k=10)
similar_docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex f

In [28]:
#Advaced  similarity search with scores(Which is important to gain knwoledge about who are closer to our answer)

In [29]:
result_scores=vector_store.similarity_search_with_score(query,k=3)
result_scores

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
  0.5676447153091431),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
  0.5676447153091431),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
  0.5676447153091431)]

In [30]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [31]:
from langchain_groq import ChatGroq

In [32]:
llm=ChatGroq(
    model="openai/gpt-oss-120b"
)

In [33]:
test_response=llm.invoke("What is an LLM?")
test_response

AIMessage(content='**LLM** stands for **Large Language Model**. It’s a type of artificial‑intelligence model that’s been trained on massive amounts of text data so it can understand and generate human‑like language. Here’s a quick rundown of what that means:\n\n| Aspect | What It Is | Why It Matters |\n|--------|------------|----------------|\n| **Scale** | “Large” refers to two things: the size of the training corpus (often billions of words) and the number of parameters (the internal numeric weights) – usually ranging from hundreds of millions to hundreds of billions. | More data and parameters generally give the model a richer grasp of language patterns, facts, and nuances. |\n| **Architecture** | Most modern LLMs use the **Transformer** architecture (introduced in the 2017 paper *Attention\u202fIs\u202fAll\u202fYou\u202fNeed*). This design relies on self‑attention mechanisms that let the model weigh the importance of each word relative to every other word in a sequence. | Transform

In [34]:
#MODERN RAG CHAIN

In [35]:
pip install chains

Note: you may need to restart the kernel to use updated packages.


c:\Users\PARAS\Downloads\krish-naik-new-course\.venv\Scripts\python.exe: No module named pip


In [36]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [37]:
#Convert vcetor stroe to retriever
retriever=vector_store.as_retriever(
    search_kwarg={"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000261E7A44CE0>, search_kwargs={})

In [38]:
###Creating an prompt template

In [39]:
from langchain_classic.prompts import ChatMessagePromptTemplate
system_prompt="""
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

{context}
"""

In [40]:
prompt=ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [41]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [42]:
##Creat a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 

In [43]:
###Create the = final RAG Chains
rag_chain=create_retrieval_chain(retriever,document_chain)

In [44]:
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000261E7A44CE0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you d

In [45]:
response=rag_chain.invoke({"input":"What is deep learning?"})

In [46]:
response

{'input': 'What is deep learning?',
 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artifici

In [47]:
response['answer']

'Deep learning is a specialized subset of machine learning that employs multi‑layered artificial neural networks. These networks automatically learn complex features and representations directly from raw data. This enables tasks such as image, speech, and language processing without manual feature engineering.'

In [48]:
#Create RAG Chai alternative using LCEL(LANGCHAIN EXPRESSION LAnguagee)

In [49]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

In [50]:
custom_prompt=ChatPromptTemplate.from_template("""
Use the follwing context to answer the question if you dont know the answers based on the context,say you dont know:
Provide specific details from the context to support your answer
                                               
Context:
{context}
Question:{question}
Answer:
""")

In [51]:
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nUse the follwing context to answer the question if you dont know the answers based on the context,say you dont know:\nProvide specific details from the context to support your answer\n\nContext:\n{context}\nQuestion:{question}\nAnswer:\n'), additional_kwargs={})])

In [52]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000261E7A44CE0>, search_kwargs={})

In [53]:
#format the output documents for the prompt
def format_docs(docs):
    return "/n/n".join(doc.page_content for doc in docs)

In [54]:
#Building the chain using LCEL
rag_chain_lcel=(
    {"context":retriever|format_docs,"question":RunnablePassthrough()}
    | custom_prompt
    | llm 
    | StrOutputParser()
)

rag_chain_lcel


{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000261E7A44CE0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nUse the follwing context to answer the question if you dont know the answers based on the context,say you dont know:\nProvide specific details from the context to support your answer\n\nContext:\n{context}\nQuestion:{question}\nAnswer:\n'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reas

In [55]:
response=rag_chain_lcel.invoke("What is deep learning?")
response

'Deep learning is a specialized subset of machine learning. It employs **multi‑layered artificial neural networks** that can automatically learn **complex features and representations directly from raw data**. This enables the system to discover intricate patterns without needing manually engineered features.'

In [56]:
query = "What is deep learning?"


docs = retriever.invoke(query)



for i, doc in enumerate(docs, 1):
    print(f"Doc {i}: {doc.page_content[:200]}")

Doc 1: DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw
Doc 2: DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw
Doc 3: DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw
Doc 4: DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw


In [57]:
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex f

In [58]:
print("Testing LCEL CHAINS")
print(f"question: {query}")
response = rag_chain_lcel.invoke("What is deep learning?")
print(response)

Testing LCEL CHAINS
question: What is deep learning?
Deep learning is a specialized subset of machine learning. According to the provided context, it “uses multi‑layered artificial neural networks to automatically learn complex features and representations from raw” data. This means deep learning models consist of many stacked neural‑network layers that can extract increasingly abstract patterns directly from unprocessed inputs without manual feature engineering.


In [59]:
# Test with another query
query2 = "What are neural networks?"
response2 = rag_chain_lcel.invoke(query2)
print(f"Question: {query2}")
print(f"Answer: {response2}")

# Test similarity search with scores
result_scores = vector_store.similarity_search_with_score(query2, k=2)
print("\nSimilarity search results with scores:")
for doc, score in result_scores:
    print(f"Score: {score:.4f} - Content: {doc.page_content[:100]}...")

Question: What are neural networks?
Answer: Neural networks are a core technology in modern artificial intelligence.  As the provided context explains, they “enable systems to recognize patterns, make predictions, and adapt to new information.”  In other words, a neural network is a computational model that processes data in a way that mimics how biological brains learn, allowing AI applications to detect regularities, forecast outcomes, and update their behavior as they encounter new data.

Similarity search results with scores:
Score: 0.5962 - Content: . Neural networks are fundamental to modern AI, enabling systems to recognize patterns, make predict...
Score: 0.5962 - Content: . Neural networks are fundamental to modern AI, enabling systems to recognize patterns, make predict...


In [60]:
#Adding new documents to the new vector store

In [61]:
vector_store

In [62]:
new_documents="""reinforcement learning 
is like teaching an agent through trial and error with rewards and penalties.
A simple example is training a robot to walk: every time it takes a step forward without falling, it gets a “reward,” and if it falls, it gets a “penalty.” 
Over time, it learns the sequence of actions that maximize rewards.
"""

In [63]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data without'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='from data without being explicitly programmed'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='in unlabeled data'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. Ma

In [64]:
new_doc=Document(
    page_content=new_documents,
    metadata={"source":"manual_addition","topic":"reinforcement_learning"})

In [65]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='reinforcement learning \nis like teaching an agent through trial and error with rewards and penalties.\nA simple example is training a robot to walk: every time it takes a step forward without falling, it gets a “reward,” and if it falls, it gets a “penalty.” \nOver time, it learns the sequence of actions that maximize rewards.\n')

In [66]:
# Split the new document
from langchain_text_splitters import TokenTextSplitter
splitter = TokenTextSplitter(chunk_size=100, chunk_overlap=10)
new_chunk = splitter.split_documents([new_doc])
new_chunk

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='reinforcement learning \nis like teaching an agent through trial and error with rewards and penalties.\nA simple example is training a robot to walk: every time it takes a step forward without falling, it gets a “reward,” and if it falls, it gets a “penalty.” \nOver time, it learns the sequence of actions that maximize rewards.\n')]

In [67]:
vector_store.add_documents(new_chunk)

['404451a9-ddf8-4b49-83ff-5f8d4c4a322b']

In [68]:
print(f"The total added chunks in the  follwing is  {len(chunks)}")
print (f"The total number fo vecor that we are working on is a follows {vector_store._collection.count ()}")

The total added chunks in the  follwing is  20
The total number fo vecor that we are working on is a follows 104


In [69]:
new_question="What are the key concepts of reinforcement learning?"
result=rag_chain_lcel.invoke(new_question)
result

'**Key concepts of reinforcement learning (as described in the context)**  \n\n1. **Agent** – the learner that must decide what to do.  \n   *The context says “reinforcement learning is like teaching an *agent* through trial and error…”.*  \n\n2. **Trial‑and‑error interaction** – the agent repeatedly tries actions and observes the outcomes.  \n   *“…through trial and error with rewards and penalties.”*  \n\n3. **Actions (or sequence of actions)** – the choices the agent can make in each step.  \n   *The robot “takes a step forward” is an example of an action.*  \n\n4. **Reward and penalty (reward signal)** – feedback that tells the agent whether a particular action was good (reward) or bad (penalty).  \n   *“Every time it takes a step forward without falling, it gets a ‘reward,’ and if it falls, it gets a ‘penalty.’”*  \n\n5. **Learning to maximize rewards** – the ultimate goal is to discover a policy (a mapping from states to actions) that yields the highest cumulative reward over tim

Advanced Rag Techniques Conversational Memory

-create)history_aware_retriver
-MessagePlaceholde
-HumanMessage/AIMessagep

In [70]:
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage,AIMessage

In [71]:
contextualize_q_system_prompt="""Given a chat history and the latest user question
which might eefernce context in the chat histroy,formulate a standalone question whihc can be understaood without the  chat history.DoNOt answer the questions just reformulate if
it is needed and otherwise return it as is"""

In [72]:
#creating an prompt that includes that chat histor
contextualize_q_prompt=ChatPromptTemplate.from_messages([
    ("system",contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}"),
])

In [73]:
#Create History aware retriever
history_aware_retriever=create_history_aware_retriever(
    llm,retriever,contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000261E7A44CE0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag

In [74]:
#Creating a new document chain with history
qa_system_prompt="""
You are an assistant fro question-answering tasks,
Use the following piences of retrieved context to answer the question
If you dont know the answer,just say that you dont know.
Use three sentences maximum and keep the answer concise
Context:{context}"""
qa_prompt=ChatPromptTemplate([
    ("system",qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}"),
])
question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)


print("Creating conversational rag  chain")

Creating conversational rag  chain


In [75]:
conversational_rag_chain=create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("Conversational rag chain")

Conversational rag chain


In [76]:
chat_history=[]
result1=conversational_rag_chain.invoke({
    "chat_history":chat_history,
    "input":'What is  machine learning?'
})
print(f"Q;What is machine learning")
print(f"A:{result1['answer']}")

Q;What is machine learning
A:Machine learning is a branch of artificial intelligence that develops algorithms and models allowing computers to automatically learn patterns and relationships from data, without being explicitly programmed for each task.


In [77]:
chat_history

[]

In [80]:
chat_history.extend([
    HumanMessage(content="What is machine learning?"),
    AIMessage(content=result1['answer'])
])

In [82]:
#Follow up question
result2=conversational_rag_chain.invoke({
    'chat_history':chat_history,
    "input":'What are its main types?'
})

In [83]:
result2

{'chat_history': [HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a branch of artificial intelligence that develops algorithms and models allowing computers to automatically learn patterns and relationships from data, without being explicitly programmed for each task.', additional_kwargs={}, response_metadata={})],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled'),
  Document(metadata={'source': 'data\\doc_0.txt'}, page_content='. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled'),
  Document